# Lesson 8.2 — Capstone: Resolved-Rate Tracker (the velocity layer)
**Module 6 · Unit 8 · Lesson 30**

`track_step(q, xi_d, dt, lam)` resolves a commanded tool twist into joint rates and integrates **open-loop** — the joint-velocity stream Module 7 will drive. NOT trajectory generation, NOT feedback control.

In [1]:
import numpy as np
def dh(th,d,a,al):
    ct,st,ca,sa=np.cos(th),np.sin(th),np.cos(al),np.sin(al)
    return np.array([[ct,-st*ca,st*sa,a*ct],[st,ct*ca,-ct*sa,a*st],[0,sa,ca,d],[0,0,0,1]])
def forward_chain(P,T,q):
    M=np.eye(4); Ms=[M.copy()]
    for i,(th0,d0,a,al) in enumerate(P):
        th,d=(th0+q[i],d0) if T[i]=="R" else (th0,d0+q[i]); M=M@dh(th,d,a,al); Ms.append(M.copy())
    return Ms
def geometric_jacobian(P,T,q):
    Ms=forward_chain(P,T,q); on=Ms[-1][:3,3]; J=np.zeros((6,len(q)))
    for i in range(len(q)):
        z=Ms[i][:3,2]; o=Ms[i][:3,3]
        if T[i]=="R": J[:3,i]=np.cross(z,on-o); J[3:,i]=z
        else: J[:3,i]=z
    return J
def Jv_planar(P,T,q): return geometric_jacobian(P,T,q)[:2,:]
def fk_xy(P,T,q): return forward_chain(P,T,q)[-1][:2,3]
P2=[(0,0,1,0),(0,0,1,0)]; T2=["R","R"]
P3=[(0,0,1,0),(0,0,1,0),(0,0,0.6,0)]; T3=["R","R","R"]   # redundant (2D task, 3 joints)
EPS=0.08   # singularity threshold on sigma_min


In [2]:
def analyze(P,T,q):
    """One SVD -> full capability report (Units 4-6 in one function)."""
    J=Jv_planar(P,T,q); U,S,Vt=np.linalg.svd(J)
    w=float(np.prod(S)); kappa=float(S[0]/max(S[-1],1e-12))
    return {"sigma":S,"w":w,"kappa":kappa,
            "axes":[(U[:,i],float(S[i])) for i in range(len(S))],
            "singular":bool(S[-1]<EPS),"sigma_min":float(S[-1]),"J":J,"U":U,"Vt":Vt}


In [3]:
def dls(J,lam):
    return J.T@np.linalg.inv(J@J.T+lam**2*np.eye(J.shape[0]))
def track_step(P,T,q,xi_d,dt,lam=0.0):
    """Resolve commanded tool twist -> joint rates, integrate open-loop one step."""
    J=Jv_planar(P,T,q)
    qd=(dls(J,lam) if lam>0 else np.linalg.pinv(J))@xi_d
    return q+qd*dt, qd


## Constant command -> straight-line tool motion, small open-loop drift

In [4]:
checks=[]
q=np.array([0.4,1.4]); dt=0.005; xi=np.array([0.25,0.0])  # modest command, mid-workspace
p0=fk_xy(P2,T2,q); steps=200
for _ in range(steps):
    q,qd=track_step(P2,T2,q,xi,dt,lam=0.0)
disp=fk_xy(P2,T2,q)-p0; expected=xi*dt*steps
print('net tool displacement:',np.round(disp,4),' expected:',np.round(expected,4),' (open-loop drift is small)')
checks.append(np.linalg.norm(disp-expected)<0.02)

net tool displacement: [ 2.5e-01 -1.0e-04]  expected: [0.25 0.  ]  (open-loop drift is small)


## A pre-supplied time-varying command is handled unchanged (generating it = Module 7)

In [5]:
q=np.array([0.3,0.9])
stream=[np.array([0.3*np.cos(0.05*k),0.3*np.sin(0.05*k)]) for k in range(150)]  # supplied externally
maxqd=0
for xi_k in stream:
    q,qd=track_step(P2,T2,q,xi_k,dt,lam=0.05); maxqd=max(maxqd,np.linalg.norm(qd))
print("tracker consumed external stream; max ||qdot||=",round(maxqd,3))
checks.append(maxqd<50 and np.all(np.isfinite(q)))
print("NOTE: the tracker only CONSUMES twists; producing the stream is Module 7's job.")
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

tracker consumed external stream; max ||qdot||= 0.807
NOTE: the tracker only CONSUMES twists; producing the stream is Module 7's job.
All checks passed.
